# 03 · Sınıflandırma Modelleri
**Breast Cancer** veri seti: 569 tümör, 30 özellik, hedef: `0 = kötü huylu`, `1 = iyi huylu`.
Lojistik Regresyon, KNN, SVM, Naive Bayes ve Karar Ağacı modellerini karşılaştırıyoruz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

## 1. Veri

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
print("Boyut:", X.shape, "| Eksik değer:", X.isna().sum().sum())
print("Sınıf dağılımı:\n", y.value_counts().rename({0: "kötü huylu", 1: "iyi huylu"}))
X.iloc[:, :6].head()

**Yorum:** Eksik değer yok. Sınıflar hafif dengesiz (%63 / %37) → train/test ayrımında `stratify=y` kullanarak oranı koruyacağız ve sadece accuracy'ye değil F1/Recall'a da bakacağız.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, col in zip(ax, ["mean radius", "mean texture", "worst concave points"]):
    sns.kdeplot(data=X.assign(hedef=y), x=col, hue="hedef", fill=True, palette=["tab:red", "tab:green"], ax=a)
plt.tight_layout(); plt.show()

**Yorum:** Özellikle `worst concave points` ve `mean radius` iki sınıfı belirgin şekilde ayırıyor → yüksek başarı bekleyebiliriz.

## 2. Train/Test ve yardımcı fonksiyon

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)   # stratify: sınıf oranlarını koru

sonuclar = {}
def degerlendir(model, ad):
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    sonuclar[ad] = {"Accuracy": accuracy_score(y_test, p), "F1": f1_score(y_test, p),
                    "Recall": recall_score(y_test, p),
                    "ROC-AUC": roc_auc_score(y_test, proba) if proba is not None else np.nan}
    print(f"{ad}: " + "  ".join(f"{k}={v:.3f}" for k, v in sonuclar[ad].items()))
    return model

## 3. Lojistik Regresyon
Adına rağmen **sınıflandırma** modelidir. Lineer kombinasyonu sigmoid fonksiyonundan geçirip olasılık üretir:
$$P(y=1)=\frac{1}{1+e^{-(\beta_0+\beta^Tx)}}$$
`C` parametresi düzenlileştirmenin tersidir (küçük C = güçlü ceza).

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = degerlendir(Pipeline([("s", StandardScaler()),
                                ("m", LogisticRegression(C=1.0, max_iter=1000))]), "Lojistik Regresyon")
print(classification_report(y_test, log_reg.predict(X_test), target_names=["kötü", "iyi"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_estimator(log_reg, X_test, y_test, display_labels=["kötü", "iyi"], cmap="Blues", ax=ax[0])
RocCurveDisplay.from_estimator(log_reg, X_test, y_test, ax=ax[1], color="tab:orange")
ax[1].plot([0, 1], [0, 1], "k--"); plt.tight_layout(); plt.show()

**Yorum:** Confusion matrix'te köşegen dışındaki hücreler hatalardır. Tıbbi problemde **kötü huylu tümörü iyi huylu sanmak** (yanlış negatif) en pahalı hatadır; bu sayıyı izlemek gerekir. ROC eğrisi sol üst köşeye ne kadar yakınsa model o kadar iyidir.

## 4. K-En Yakın Komşu (KNN)
Yeni noktaya en yakın **k** komşunun çoğunluk sınıfını atar. Mesafe tabanlı olduğu için **ölçekleme şarttır**.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

k_degerleri = range(1, 31)
cv_skor = [cross_val_score(Pipeline([("s", StandardScaler()), ("m", KNeighborsClassifier(k))]),
                           X_train, y_train, cv=5, scoring="f1").mean() for k in k_degerleri]
en_iyi_k = k_degerleri[int(np.argmax(cv_skor))]
plt.figure(figsize=(8, 3.5)); plt.plot(k_degerleri, cv_skor, "o-", color="tab:purple")
plt.axvline(en_iyi_k, color="r", ls="--"); plt.title(f"KNN – k seçimi (en iyi k={en_iyi_k})"); plt.xlabel("k"); plt.ylabel("CV F1"); plt.show()

degerlendir(Pipeline([("s", StandardScaler()), ("m", KNeighborsClassifier(en_iyi_k))]), "KNN")

**Yorum:** k küçükse model gürültüye duyarlıdır (overfitting), k büyükse sınırlar fazla yumuşar (underfitting). k'yı test setiyle değil, **train üzerinde çapraz doğrulama** ile seçtik.

## 5. Destek Vektör Makineleri (SVM)
Sınıflar arasındaki **marjini en büyük** ayırıcı düzlemi bulur. `kernel="rbf"` ile doğrusal olmayan sınırlar çizer.

In [ ]:
from sklearn.svm import SVC
degerlendir(Pipeline([("s", StandardScaler()), ("m", SVC(kernel="rbf", C=1, probability=True, random_state=42))]), "SVM (RBF)")
degerlendir(Pipeline([("s", StandardScaler()), ("m", SVC(kernel="linear", C=0.1, probability=True, random_state=42))]), "SVM (Linear)")

**Yorum:** Bu veri neredeyse doğrusal ayrılabilir olduğu için lineer ve RBF çekirdek benzer sonuç verir.

## 6. Naive Bayes
Bayes teoremine dayanır, özelliklerin **birbirinden bağımsız** olduğunu varsayar. Çok hızlıdır; metin sınıflandırmada (spam) `MultinomialNB` çok başarılıdır.

In [ ]:
from sklearn.naive_bayes import GaussianNB
degerlendir(GaussianNB(), "Naive Bayes")

**Yorum:** Bu veride özellikler birbiriyle güçlü korelasyonlu (radius, perimeter, area…), yani bağımsızlık varsayımı ihlal ediliyor; buna rağmen makul sonuç verir ama genellikle diğerlerinin biraz gerisinde kalır.

## 7. Karar Ağacı
Veriyi "evet/hayır" sorularıyla dallara ayırır. Yorumlanabilir ama tek başına overfitting'e yatkındır (`max_depth` ile sınırlayın).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
agac = degerlendir(DecisionTreeClassifier(max_depth=3, random_state=42), "Karar Ağacı")
plt.figure(figsize=(20, 8))
plot_tree(agac, feature_names=X.columns, class_names=["kötü", "iyi"], filled=True, fontsize=9)
plt.show()

**Yorum:** Ağacın ilk bölünmesi en ayırt edici özelliği gösterir. Her kutuda `gini` (saflık), örnek sayısı ve sınıf dağılımı yer alır — bir doktora modelin kararını adım adım açıklayabilirsiniz.

## 8. Karşılaştırma

In [ ]:
karsilastirma = pd.DataFrame(sonuclar).T.sort_values("F1", ascending=False)
display(karsilastirma.round(3))
karsilastirma[["Accuracy", "F1", "ROC-AUC"]].plot.barh(figsize=(9, 5), xlim=(0.85, 1.0), colormap="Set2")
plt.title("Sınıflandırma Modelleri – Test Performansı"); plt.tight_layout(); plt.show()

## Özet
### Data Analysis Key Findings
- Ölçeklenmiş lineer modeller (Lojistik Regresyon, SVM) bu veri setinde genellikle en yüksek F1 ve ROC-AUC (≈0.98–0.99) değerlerine ulaşır.
- Tek karar ağacı en yorumlanabilir model, ancak performansı biraz daha düşük.
- Mesafe/marjin tabanlı modellerde (KNN, SVM) ölçekleme zorunludur.
### Insights or Next Steps
- Tek ağacın zayıflığını gidermek için **04_Ensemble_Modeller** notebook'una geçin.